# RAG workshop: Введення та основні поняття

Що буде сьогодні:
Для чого
Термінологія
Крок по кроку 


Що потрібно щоб відтворити код в ноутбуці: 
**Потрібен:** OpenAI API ключ (використовуємо `text-embedding-3-*` + `gpt-*`)


## 00 Introduction

![Fine-tuning vs RAG — навички vs знання](https://raw.githubusercontent.com/NatalyUA/RAG-workshop/main/RAG_intro.png)

### Що таке RAG?

**RAG (Retrieval-Augmented Generation)** — підхід, у якому перед відповіддю LLM спочатку **шукають** релевантні фрагменти в зовнішньому корпусі (ваші PDF, таблиці, нотатки), а потім модель **генерує** текст, спираючись на знайдений контекст. Типова схема: **запит → retrieval → підстановка контексту в промпт → відповідь**.


### Why&When, How, For What

![Mind map — RAG: чому, як, які кейси](https://raw.githubusercontent.com/NatalyUA/RAG-workshop/main/RAG_mindmap.png)

**HOW ланцюг:** 
**chunking** (нарізка) → **embedding** (текст → вектори) → **vector store** (зберігання) → **retrieval** (пошук релевантних фрагментів) → **augmentation** (контекст у промпті) → **generation** (відповідь LLM).

In [1]:
!pip install openai chromadb python-dotenv -q

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import numpy as np

# 🔑 Ключ з файлу .env у корені проєкту (рядок OPENAI_API_KEY=...)
for _root in [Path.cwd(), *Path.cwd().parents]:
    _env = _root / ".env"
    if _env.is_file():
        load_dotenv(_env)
        break
else:
    load_dotenv()

# Альтернатива для Colab: беремо ключ із Secrets (ліва панель -> key icon)
if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata

        _colab_key = userdata.get("OPENAI_API_KEY")
        if _colab_key:
            os.environ["OPENAI_API_KEY"] = _colab_key
    except Exception:
        pass

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "Немає OPENAI_API_KEY. Додайте .env у корінь проєкту, змінну середовища або Colab Secret OPENAI_API_KEY."
    )

client = OpenAI()
EMBED_MODEL = "text-embedding-3-large"  # дешева, швидка, 1536 вимірів

## Допоміжні функції і їх використання 

In [3]:
# Векторизація (Embedding): текст → числовий вектор
def embed(text: str) -> np.ndarray:
    """Один рядок тексту → один вектор чисел."""
    response = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(response.data[0].embedding)

# Косинусна схожість (cosine similarity): порівняння векторів
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [4]:
# Приклад використання. 
word1 = "їдальня"
word2 = "ресторан"

print(cosine_sim(embed(word1), embed(word2)))

# 📝 Завдання #1: 
# Спробуйте власний запит. Що більше схоже на "їдальня" i "кафе" чи "шашлична"? 
# Чи може "аеропорт" бути близьким до "шашличної"?


0.5039990245604682


---
## Етап 1: Embedding + Similarity 

**Ідея:** текст → вектор (список чисел), де схожий зміст = близькі числа.



In [5]:

# Знайти слова, що схожі на "queen" серед списку слів: : embedding + similarity. 

phrase_for_compare = "queen"
phrases = [
    "queen",
    "королева",
    "king",
    "король",
    "man",
    "woman",
    "prince",
    "princess",
]

vecs = embed(phrase_for_compare)
print(f"Схожість з {phrase_for_compare}  :\n")

scores = []
# Для кожної фрази рахуємо схожість із запитом і зберігаємо пару (фраза, показник схожості)
for phrase in phrases:
    similarity = cosine_sim(vecs, embed(phrase))
    scores.append((phrase, similarity))
    
for phrase, score in sorted(scores, key=lambda x: -x[1]):
    bar = "█" * int(score * 25)
    print(f"  {score:.3f}  {bar:<25}  {phrase}")

# 📝 Завдання #2:     
# a) Порівняйте всі слова в списку зі словом "king"
# b) Додайте до списку ще кілька слів, наприклад "emperor", "president", "builder", 
#    "artist" або інші на ваш вибір, подивіться на їх показники схожості

Схожість з queen  :

  1.000  █████████████████████████  queen
  0.565  ██████████████             королева
  0.552  █████████████              king
  0.413  ██████████                 woman
  0.405  ██████████                 princess
  0.399  █████████                  prince
  0.394  █████████                  король
  0.307  ███████                    man




--- 
🧐 Над ембеддінгами можливі і інші операції, не тільки порівняння. Класичний приклад: у просторі ембеддингів слів виконується **king − man + woman ≈ queen**, про що можна почитати в цікавій науковій статті **Jay Alammar у «The Illustrated Word2Vec».** 



In [6]:
# Пошук по словосполученям: embedding + similarity
phrase_to_compare = "dog friendly"

phrases = [
    "pet-friendly",
    "dogs/cats allowed",
    "dog-friendly",
    "з собакою дозволяється",
    "з кішкою дозволяється",
    "проживання з собакою заборонено",
    "проживання з домашніми улюбленцями дозволено",
    "проживання з домашніми улюбленцями заборонено",
    "маленькі тварини дозволені",# non-english
    "можна з домашніми улюбленцями", # non-english
    "сучасний ремонт", # нерелевантне, # non-english
    "тварини заборонені", # non-english, антонім
    "animals forbidden",    # антонім
    "free WiFi included",      # нерелевантне
    "parking available nearby", # нерелевантне
]

vecs = embed(phrase_to_compare)

scores = []
for phrase in phrases:
    similarity = cosine_sim(vecs, embed(phrase))
    scores.append((phrase, similarity))
    
for phrase, score in sorted(scores, key=lambda x: -x[1]):
    bar = "█" * int(score * 25)
    print(f"  {score:.3f}  {bar:<25}  {phrase}")
        

# Завдання #3: 
# Спробуйте замінити пошукову фразу на українську, наприклад "можна з собакою"
# Зверніть увагу як зміниться порядок відсортованого по схожесті списку  



  0.954  ███████████████████████    dog-friendly
  0.850  █████████████████████      pet-friendly
  0.667  ████████████████           dogs/cats allowed
  0.514  ████████████               з собакою дозволяється
  0.485  ████████████               animals forbidden
  0.450  ███████████                проживання з домашніми улюбленцями дозволено
  0.436  ██████████                 можна з домашніми улюбленцями
  0.416  ██████████                 проживання з собакою заборонено
  0.401  ██████████                 маленькі тварини дозволені
  0.382  █████████                  parking available nearby
  0.380  █████████                  free WiFi included
  0.364  █████████                  проживання з домашніми улюбленцями заборонено
  0.341  ████████                   з кішкою дозволяється
  0.313  ███████                    тварини заборонені
  0.144  ███                        сучасний ремонт


--- 
🧐 
### Важливе зауваження: 
Косинусна схожість ембеддингів — корисний інструмент, але не «магія». Якість залежить від того, що саме ви порівнюєте: один довгий документ і одне коротке речення дають різну «концентрацію» змісту в одному векторі; змішані мови, шум, шаблонний текст або рідкісні формулювання теж зміщують оцінку. Тому іноді важливо свідомо підбирати довжину й роль фрагментів (речення, абзац, чанк), а не сліпо довіряти одному числу.

**Практичний напрям покращення** — комбінувати різні типи збігів: порівняння слово до слова (лексика, BM25 тощо), фраза до фрази (семантика через ембеддинги), і гібрид, де обидва сигнали зважуються разом. Є й складніші підходи (reranking, навчені скорери, крос-енкодери тощо) — вони виходять за рамки цього воркшопу,

---
## 📦 Етап 2: Chunking — оголошення про квартиру 🏠

**Проблема:**
- LLM не може з'їсти великий текст цілком — є ліміт токенів.
- Similarity на довгих текстах розмивається.
- Коротке питання примусить аналізувати багатосторінковий документ.

**Рішення:** нарізати текст на шматки і зберігати їх окремо.


In [7]:
query = "є паркінг?"

listing = """
Здається простора 2-кімнатна квартира в новому ЖК бізнес-класу на Печерську.
Загальна площа — 72 м², житлова — 45 м². Висота стель 3 м, панорамні вікна.
Квартира повністю мебльована та обладнана технікою: кухонний гарнітур, вбудована техніка,
двоспальне ліжко, великий гардероб, пральна машина, посудомийка.
Є окремий кабінет — зручно для роботи вдома.
Інтернет оптоволоконний, швидкість 1 Гбіт/с — включено у вартість.
Опалення автономне, лічильники на все — платите лише за спожите.
Квартира має підземний паркінг, який не входить у вартість оренди —
оплачується окремо, 1500 грн/міс. Місце відеоспостереження, охорона цілодобово.
Будинок — закрита територія, консьєрж, дитячий майданчик у дворі.
Тварини — за домовленістю, невеликі породи OK. Неподалік є зручний парк для прогулянок.
Оренда: 28 000 грн/міс. + депозит 1 місяць. Комунальні послуги — окремо.
"""

listing_no_parking = """
Здається простора 2-кімнатна квартира в новому ЖК бізнес-класу на Печерську.
Загальна площа — 72 м², житлова — 45 м². Висота стель 3 м, панорамні вікна.
Квартира повністю мебльована та обладнана технікою: кухонний гарнітур, вбудована техніка,
двоспальне ліжко, великий гардероб, пральна машина, посудомийка.
Є окремий кабінет — зручно для роботи вдома.
Інтернет оптоволоконний, швидкість 1 Гбіт/с — включено у вартість.
Опалення автономне, лічильники на все — платите лише за спожите.
Місце відеоспостереження, охорона цілодобово. 
Будинок — закрита територія, консьєрж, дитячий майданчик у дворі.
Тварини — за домовленістю, невеликі породи OK. Неподалік є зручний парк для прогулянок.
Оренда: 28 000 грн/міс. + депозит 1 місяць. Комунальні послуги — окремо.
"""

# Порівнюємо ЦІЛІ оголошення (без розбиття на чанки)
q_vec_parking = embed(query)

score_with    = cosine_sim(q_vec_parking, embed(listing.strip()))
score_without = cosine_sim(q_vec_parking, embed(listing_no_parking.strip()))

print("=" * 60)
print(f'Запит: «{query}» — порівнюємо ЦІЛІ оголошення')
print("=" * 60)
print(f"  З паркінгом  (слово є в тексті):  score = {score_with:.3f}")
print(f"  Без паркінгу (слова немає):        score = {score_without:.3f}")
print(f"  Різниця:                           {abs(score_with - score_without):.3f}")
print()


Запит: «є паркінг?» — порівнюємо ЦІЛІ оголошення
  З паркінгом  (слово є в тексті):  score = 0.338
  Без паркінгу (слова немає):        score = 0.326
  Різниця:                           0.013



---
### Зверніть увагу: 
Різниця мізерна — один факт 'тоне' в довгому векторі.  Саме тому текст розбивають на чанки. Розбиття може бути просто по довжині, а може бути більш складним з урахуванням структури документу (по роздіалах, сторінках), або семантично - по параграфах, закінчених реченях.  

## Допоміжна функцій розбиття на фрагменти і приклад використання 

In [8]:
def chunker(text: str, chunk_size: int = 256, overlap: int = 64) -> list[str]:
    """Нарізає текст на фрагменти з перекриттям (overlap)."""
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start : start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunker(listing, 128, 32) 
print(f"Нарізано на {len(chunks)} chunks:\n")
for i, c in enumerate(chunks):
    print(f"[{i+1}] {c}\n")

Нарізано на 9 chunks:

[1] Здається простора 2-кімнатна квартира в новому ЖК бізнес-класу на Печерську.
Загальна площа — 72 м², житлова — 45 м². Висота ст

[2] 2 м², житлова — 45 м². Висота стель 3 м, панорамні вікна.
Квартира повністю мебльована та обладнана технікою: кухонний гарнітур,

[3] ана технікою: кухонний гарнітур, вбудована техніка,
двоспальне ліжко, великий гардероб, пральна машина, посудомийка.
Є окремий к

[4] машина, посудомийка.
Є окремий кабінет — зручно для роботи вдома.
Інтернет оптоволоконний, швидкість 1 Гбіт/с — включено у варті

[5] ість 1 Гбіт/с — включено у вартість.
Опалення автономне, лічильники на все — платите лише за спожите.
Квартира має підземний пар

[6] жите.
Квартира має підземний паркінг, який не входить у вартість оренди —
оплачується окремо, 1500 грн/міс. Місце відеоспостереж

[7] 00 грн/міс. Місце відеоспостереження, охорона цілодобово.
Будинок — закрита територія, консьєрж, дитячий майданчик у дворі.
Твар

[8] дитячий майданчик у дворі.
Тварини —

In [9]:
# Без overlap:
chunks_without_overlap = chunker(listing, 128, 0)  
print(f"Нарізано на {len(chunks_without_overlap)} chunks:\n")
for i, c in enumerate(chunks_without_overlap):
    print(f"[{i+1}] {c}\n")

Нарізано на 7 chunks:

[1] Здається простора 2-кімнатна квартира в новому ЖК бізнес-класу на Печерську.
Загальна площа — 72 м², житлова — 45 м². Висота ст

[2] ель 3 м, панорамні вікна.
Квартира повністю мебльована та обладнана технікою: кухонний гарнітур, вбудована техніка,
двоспальне л

[3] іжко, великий гардероб, пральна машина, посудомийка.
Є окремий кабінет — зручно для роботи вдома.
Інтернет оптоволоконний, швидк

[4] ість 1 Гбіт/с — включено у вартість.
Опалення автономне, лічильники на все — платите лише за спожите.
Квартира має підземний пар

[5] кінг, який не входить у вартість оренди —
оплачується окремо, 1500 грн/міс. Місце відеоспостереження, охорона цілодобово.
Будино

[6] к — закрита територія, консьєрж, дитячий майданчик у дворі.
Тварини — за домовленістю, невеликі породи OK. Неподалік є зручний п

[7] арк для прогулянок.
Оренда: 28 000 грн/міс. + депозит 1 місяць. Комунальні послуги — окремо.



In [10]:
# 🚨 Навіщо overlap: приклад з паркінгом

# Обираємо 3 найрелевантніші чанки до запиту "є паркінг?" (розбиття з оверлаппінгом)
scores = []
for chunk in chunks:
    similarity = cosine_sim(q_vec_parking, embed(chunk))
    scores.append((chunk, similarity))

for chunk, score in sorted(scores, key=lambda x: -x[1])[:3]:
    print(f"  {score:.3f}  {chunk}")    

  0.544  жите.
Квартира має підземний паркінг, який не входить у вартість оренди —
оплачується окремо, 1500 грн/міс. Місце відеоспостереж
  0.464  дитячий майданчик у дворі.
Тварини — за домовленістю, невеликі породи OK. Неподалік є зручний парк для прогулянок.
Оренда: 28 0
  0.449  ість 1 Гбіт/с — включено у вартість.
Опалення автономне, лічильники на все — платите лише за спожите.
Квартира має підземний пар


In [11]:
# 🚨 Навіщо overlap: приклад з паркінгом. Частина 2

# Обираємо 3 найрелевантніші чанки до запиту "є паркінг?" (розбиття БЕЗ оверлаппінгом)
scores = []
for chunk in chunks_without_overlap :
    similarity = cosine_sim(q_vec_parking, embed(chunk))
    scores.append((chunk, similarity))

for chunk, score in sorted(scores, key=lambda x: -x[1])[:3]:
    print(f"  {score:.3f}  {chunk}")   

# 📝 Завдання #4: 
# a) Виведіть не 3 найрелевантніші чанки, а всі, подивіться на показник схожості до втраченого фрагменту 
# b) Що зміниться в коді, якщо треба обрати не три чанкі, в ті, для яких показник схожості більше певного порога, наприклад 0.4?

  0.454  к — закрита територія, консьєрж, дитячий майданчик у дворі.
Тварини — за домовленістю, невеликі породи OK. Неподалік є зручний п
  0.449  ість 1 Гбіт/с — включено у вартість.
Опалення автономне, лічильники на все — платите лише за спожите.
Квартира має підземний пар
  0.433  арк для прогулянок.
Оренда: 28 000 грн/міс. + депозит 1 місяць. Комунальні послуги — окремо.


---
## 🗄️ Етап 3: Vector Store — Полісі нерухомості 🏠

До цього ми тримали вектори в пам'яті (numpy).  
**Vector Store** = база даних для векторів: зберігає на диску, шукає за мілісекунди.

Використовуємо **ChromaDB** — найпростіший варіант для старту.

### Що треба знати про ChromaDB
- `Client()` — все в оперативній пам'яті: зручно для демо, але зникає при перезапуску
- `PersistentClient(path=...)` — зберігає на диск, дані лишаються між сесіями
- Альтернатива локально: **Qdrant** — швидший і з hybrid search із коробки
- У продакшні: Qdrant Cloud, Pinecone, Weaviate — або pgvector якщо вже є PostgreSQL

- **Важливо про міру схожості**: при hnsw:space: "cosine" ChromaDB повертає cosine distance (від 0 до 2), а не схожість. Схожість = 1 - distance. Тому найрелевантніший чанк має найменшу дистанцію і стоїть першим.


In [12]:
import chromadb

chroma = chromadb.Client()  # in-memory для демо

# Cтворюємо колекцію. Увага - один раз створюємо колекцію, далі додаємо до неї дані, якщо знов виконати цю команду, то буде повідомлення про помилку:
# InternalError: Collection [realty_policies] already exists

collection = chroma.create_collection("realty_policies", metadata={"hnsw:space": "cosine"})

In [13]:


# Полісі нерухомості — серйозні (і не дуже)
policies = [
    """Тварини в оренді.
Кіт — завжди так. Пес до 10 кг — з письмовою згодою сусідів. Пес понад 10 кг — лише якщо вміє
тихо дивитися серіали і не гавкає на доставку. Папуга — заборонено, якщо знає матюки.""",

    """Шум і вечірки.
Вечірки дозволені до 23:00. Після 23:00 — виключно тиха дискотека у навушниках.
Перфоратор та дриль — з 10:00 до 18:00 у будні. Хто порушує — слухає перфоратор від сусідів о 7:00.""",

    """Депозит і повернення.
Депозит становить 1 місяць оренди і повертається протягом 30 днів після виїзду.
Не повертається, якщо залишено більше 3 дір від картин або кіт погриз кут дивана.""",

    """Ремонт і зміни.
Будь-який ремонт потребує письмової згоди орендодавця. Стіни можна фарбувати лише в нейтральні кольори.
Яскравий помаранчевий — заборонений, навіть якщо це "для натхнення".""",

    """Перевірка квартири орендодавцем.
Орендодавець попереджає про візит за 24 години. "Просто проходив повз" — не є поважною причиною.
Перевірка раз на місяць, без фотографування особистих речей орендаря.""",

    """Купівля квартири — документи.
Для угоди потрібно: паспорт, ідентифікаційний код і довідка про доходи.
А також — фото вашого обличчя, коли ви дізналися реальну ціну. Нотаріус зберігає всі копії.""",

    """Купівля квартири — строки.
Угода підписується у нотаріуса. Середній строк оформлення — 2–4 тижні.
Якщо банк, іпотека і забудовник не сперечаються — може бути і швидше. Так буває рідко.""",

    """Паркінг.
Паркінг бронюється за принципом: першим прийшов — першим зайняв.
Велосипеди паркувати у спеціальному місці, не в ліфті — навіть якщо він дуже дорогий.""",
]

# Chunking: нарізаємо кожен полісі на шматки
all_chunks = []
for doc in policies:
    chunks = chunker(doc, chunk_size=180, overlap=20)
    all_chunks.extend(chunks)

print(f"Документів: {len(policies)}, чанків після нарізки: {len(all_chunks)}\n")

# Embedding і додавання до ChromaDB
for i, chunk in enumerate(all_chunks):
    vec = embed(chunk).tolist()
    collection.add(ids=[str(i)], documents=[chunk], embeddings=[vec])

print(f"✅ Додано {collection.count()} чанків у ChromaDB")


Документів: 8, чанків після нарізки: 15

✅ Додано 15 чанків у ChromaDB


In [14]:
query = "до котрої години можна шуміти?"
q_vec = embed(query).tolist()
res = collection.query(query_embeddings=[q_vec], n_results=5)

# ChromaDB вже повертає відсортованими: нагадуємо що найрелевантніший чанк має найменшу дистанцію і стоїть першим
results = zip(res["distances"][0], res["documents"][0])
for dist, doc in results:
    print(f"[{dist:.3f}] {doc}")
    print()
    

[0.298] Шум і вечірки.
Вечірки дозволені до 23:00. Після 23:00 — виключно тиха дискотека у навушниках.
Перфоратор та дриль — з 10:00 до 18:00 у будні. Хто порушує — слухає перфоратор від с

[0.493] хає перфоратор від сусідів о 7:00.

[0.606] Тварини в оренді.
Кіт — завжди так. Пес до 10 кг — з письмовою згодою сусідів. Пес понад 10 кг — лише якщо вміє
тихо дивитися серіали і не гавкає на доставку. Папуга — заборонено,

[0.606] апуга — заборонено, якщо знає матюки.

[0.632] Перевірка квартири орендодавцем.
Орендодавець попереджає про візит за 24 години. "Просто проходив повз" — не є поважною причиною.
Перевірка раз на місяць, без фотографування особис



---
## 🧠 Етап 4: Augmentation + Generation — Полісі нерухомості

Фінальний крок RAG:
1. Знайти релевантні chunks *(вже вміємо)*
2. **Вставити їх у промпт** як контекст
3. LLM відповідає **тільки на основі цього контексту**


In [15]:
# Шаблон промпту — зберігається окремо, підставляємо пізніше
PROMPT_TEMPLATE = """Ти асистент агентства нерухомості. Відповідай тільки на основі наданого контексту.
Якщо відповіді немає в контексті — скажи, що не знаєш.

Контекст:
{context}

Питання: {question}
Відповідь:"""

# Крок 1: запит користувача
question = "Чи можна тримати собаку у квартирі?"

# Крок 2: шукаємо релевантні чанки у векторній базі
q_vec = embed(question).tolist()
res = collection.query(query_embeddings=[q_vec], n_results=3)
chunks = res["documents"][0]

# Крок 3: формуємо контекст із знайдених чанків
context = "\n---\n".join(chunks)

# Крок 4: підставляємо змінні в шаблон
prompt = PROMPT_TEMPLATE.format(context=context, question=question)

print("📋 Промпт який піде в LLM:")
print(prompt)
print()

# Крок 5: викликаємо LLM
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
)

# Крок 6: виводимо відповідь
print("🤖", response.choices[0].message.content)


📋 Промпт який піде в LLM:
Ти асистент агентства нерухомості. Відповідай тільки на основі наданого контексту.
Якщо відповіді немає в контексті — скажи, що не знаєш.

Контекст:
Тварини в оренді.
Кіт — завжди так. Пес до 10 кг — з письмовою згодою сусідів. Пес понад 10 кг — лише якщо вміє
тихо дивитися серіали і не гавкає на доставку. Папуга — заборонено,
---
Перевірка квартири орендодавцем.
Орендодавець попереджає про візит за 24 години. "Просто проходив повз" — не є поважною причиною.
Перевірка раз на місяць, без фотографування особис
---
кіт погриз кут дивана.

Питання: Чи можна тримати собаку у квартирі?
Відповідь:

🤖 Собаку можна тримати у квартирі, якщо вона важить до 10 кг з письмовою згодою сусідів, або якщо вона понад 10 кг і вміє тихо дивитися серіали та не гавкає на доставку.


In [16]:
# Завдання #6. Спробуйте ще кілька питань
#
#   "Коли повертають депозит?",
#   "До котрої години можна робити ремонт?",
#   "Які документи потрібні для купівлі?",


In [17]:
# 🚨 Питання поза базою — модель не вигадує
question = "Яка мінімальна зарплата для отримання іпотеки?"

q_vec = embed(question).tolist()
res = collection.query(query_embeddings=[q_vec], n_results=3)
context = "\n---\n".join(res["documents"][0])

prompt = PROMPT_TEMPLATE.format(context=context, question=question)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
)

print(f"❓ {question}")
print(f"🤖 {response.choices[0].message.content}")
print()
print("💡 Модель не вигадала — бо в контексті цього немає.")
print("   Це і є головна перевага RAG: відповідь лише з вашої бази знань.")


❓ Яка мінімальна зарплата для отримання іпотеки?
🤖 Не знаю.

💡 Модель не вигадала — бо в контексті цього немає.
   Це і є головна перевага RAG: відповідь лише з вашої бази знань.


## RAG end-to-end
---
![RAG end-to-end pipeline](https://raw.githubusercontent.com/NatalyUA/RAG-workshop/main/RAG_end-to-end_pipeline.png)
---

### Що далі — реальний кейс:
- Завантаження документів
- Формування векторного індексу 
- Промпт інжінірінг
- Streamlit інтерфейс

In [ ]:
import os
from telegram.ext import ApplicationBuilder, MessageHandler, filters

# ... ваш embed, collection, PROMPT_TEMPLATE, client вже готові

BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")

# Альтернатива для Colab: беремо BOT токен із Secrets (ліва панель -> key icon)
if not BOT_TOKEN:
    try:
        from google.colab import userdata

        BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
    except Exception:
        pass

if not BOT_TOKEN:
    raise ValueError(
        "Set TELEGRAM_BOT_TOKEN in environment variables or in Colab Secret TELEGRAM_BOT_TOKEN"
    )

# Якщо комірка запускається повторно, коректно зупиняємо попередній інстанс.
if "app" in globals() and app is not None:
    try:
        await app.updater.stop()
    except Exception:
        pass
    try:
        await app.stop()
    except Exception:
        pass
    try:
        await app.shutdown()
    except Exception:
        pass

async def handle(update, context):
    question = update.message.text
    q_vec = embed(question).tolist()
    res = collection.query(query_embeddings=[q_vec], n_results=3)
    ctx = "\n---\n".join(res["documents"][0])
    prompt = PROMPT_TEMPLATE.format(context=ctx, question=question)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    await update.message.reply_text(response.choices[0].message.content)

app = ApplicationBuilder().token(BOT_TOKEN).build()
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle))

# Jupyter/VS Code Notebook: стартуємо без app.run_polling(),
# щоб не закривати вже активний event loop.
await app.initialize()
await app.start()
await app.updater.start_polling()
print("Bot is running. To stop: await app.updater.stop(); await app.stop(); await app.shutdown()")

Bot is running. To stop: await app.updater.stop(); await app.stop(); await app.shutdown()
